In [ ]:
import sys
sys.path.insert(0, '/home/playdata2/final_pj/energy-platform')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import STL
from scripts.eda_stl_electric import run_stl_eda, meter_config, build_input_series, configure_matplotlib, load_raw_meter_data
%matplotlib inline
configure_matplotlib()
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

FEATURE_UNITS = {
    'P': 'W',
    'PF': 'ratio',
    'I1': 'A',
    'Ta': 'degC',
    'Igm': 'W/m2',
    'f': 'Hz',
    'Q': 'var',
    'Tdiff': 'degC',
    'Tvl': 'degC',
    'Trl': 'degC',
    'qv': 'm3/h',
}


In [ ]:
def plot_stl(result, anomaly_mask, title, sigma=3, unit=None):
    observed = result.observed
    trend = result.trend
    seasonal_ = result.seasonal
    residual = result.resid

    if len(observed) == 0 or observed.index.empty:
        print('빈 STL 결과라 플롯을 생략합니다.')
        return

    mean = residual.mean()
    std = residual.std()
    upper = mean + sigma * std
    lower = mean - sigma * std

    fig, axes = plt.subplots(4, 1, figsize=(18, 10), sharex=True)
    display_title = title if not unit else f'{title} [{unit}]'
    fig.suptitle(display_title, fontsize=13, fontweight='bold')

    axes[0].plot(observed.index, observed, lw=0.8, color='steelblue')
    axes[0].set_ylabel('Observed')
    axes[0].set_title('Observed', loc='left', fontsize=11, fontweight='bold')

    axes[1].plot(trend.index, trend, lw=0.8, color='orange')
    axes[1].set_ylabel('Trend')
    axes[1].set_title('Trend', loc='left', fontsize=11, fontweight='bold')

    axes[2].plot(seasonal_.index, seasonal_, lw=0.8, color='green')
    axes[2].set_ylabel('Seasonal')
    axes[2].set_title('Seasonal', loc='left', fontsize=11, fontweight='bold')

    axes[3].plot(residual.index, residual, lw=0.8, color='gray')
    axes[3].axhline(upper, color='red', linestyle='--', lw=1, label=f'+{sigma}σ')
    axes[3].axhline(lower, color='red', linestyle='--', lw=1, label=f'-{sigma}σ')
    axes[3].axhline(mean, color='black', linestyle='--', lw=0.8)
    axes[3].scatter(
        residual[anomaly_mask].index,
        residual[anomaly_mask],
        color='red', s=5, zorder=5,
        label=f'이상 ({anomaly_mask.sum()}건)'
    )
    axes[3].set_ylabel('Residual')
    axes[3].set_title('Residual', loc='left', fontsize=11, fontweight='bold')
    axes[3].legend(loc='upper right', fontsize=8)

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y', tz=observed.index.tz))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

    x_min = observed.index.min()
    x_max = observed.index.max()
    if pd.notna(x_min) and pd.notna(x_max):
        axes[-1].set_xlim(x_min, x_max)

    plt.tight_layout()
    plt.show()


## H1.Z10 — Test
변수: P, PF, I1

In [ ]:
meter_urn = 'H1.Z10'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))


## H1.Z13 — HVAC
변수: P, PF, I1, Ta

In [ ]:
meter_urn = 'H1.Z13'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))


## H1.Z16 — CM1
변수: P, PF, I1, Ta
특이사항: 2018~2019 비가동 구간이 길고, 2023년 이후 부하 급증 구간이 큼

In [ ]:
meter_urn = 'H1.Z16'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))


## H1.Z20 — CHP 전기 생산 메인
변수: P, PF
특이사항: 발전 계량기라서 P 음수도 정상일 수 있음

In [ ]:
meter_urn = 'H1.Z20'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))


## H2.T.Z33 — Distribution
변수: P, PF, I1, Igm

In [ ]:
meter_urn = 'H2.T.Z33'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))


## H2.Z35 — Transformer
변수: P, PF, I1, Igm
특이사항: 결측률을 출력하고, 높으면 해석 주의 메시지만 남기고 계속 진행

In [ ]:
meter_urn = 'H2.Z35'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))


## H2.Z64 — Server
변수: P, PF, I1, f

In [ ]:
meter_urn = 'H2.Z64'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))


## H2.Z68 — Ventilation
변수: P, PF, I1, Q

In [ ]:
meter_urn = 'H2.Z68'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))


## H2.ZE64 — Server redundant
변수: P, PF, I1
특이사항: 2022-11 이후 데이터만 존재

In [ ]:
meter_urn = 'H2.ZE64'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))


## H4.Z50 — Office distribution
변수: P, PF, I1, Ta

In [ ]:
meter_urn = 'H4.Z50'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))


## V.Z84 — PV 그룹1&2 메인
변수: P, PF, Igm
특이사항: 발전 계량기라서 P 음수도 정상일 수 있음

In [ ]:
meter_urn = 'V.Z84'
df = load_raw_meter_data(meter_urn)

for col in meter_config[meter_urn]:
    if col not in df.columns:
        print(f'{col} 컬럼 없음, skip')
        continue
    null_ratio = df[col].isnull().mean() * 100
    print(f'{col} 결측률: {null_ratio:.1f}%')
    if null_ratio > 50:
        print('  → 결측률 높음, STL 결과 해석 주의')
    result, mask = run_stl_eda(build_input_series(df, col), f'{meter_urn} - {col}', sigma=3)
    plot_stl(result, mask, f'{meter_urn} - {col} STL 분해 (±3σ)', sigma=3, unit=FEATURE_UNITS.get(col))
